In [13]:
import torch

In [2]:
torch.zeros(4, 8, 16).dim()

3

In [3]:
import math

import torch
from einops import einsum


class Linear(torch.nn.Module):
    weights: torch.Tensor

    def __init__(self, in_features: int, out_features: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.weights = torch.nn.parameter.Parameter(torch.empty(out_features, in_features, device=device, dtype=dtype))
        std = math.sqrt(2 / (in_features + out_features))
        torch.nn.init.trunc_normal_(tensor=self.weights, mean=0, std=std, a=-3 * std, b=3 * std)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return einsum(x, self.weights, "... in, out in -> ... out")

In [4]:
linear = Linear(1024, 1024)
linear(torch.randn(1024, 1024))

list(linear.state_dict().keys())

/Users/pjurczen/repo/cs336-assignment1-basics/.venv/lib/python3.13/site-packages/einops/parsing.py:72: RuntimeWarning: It is discouraged to use axes names that are keywords: in
  is_axis_name, reason = self.check_axis_name_return_reason(x, allow_underscore=allow_underscore)


['weights']

In [5]:
import torch
import math
from einops import einsum

class Embedding(torch.nn.Module):

    embeddings: torch.Tensor # (num_embeddings, embedding_dim)

    def __init__(self, num_embeddings: int, embedding_dim: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.embeddings = torch.nn.parameter.Parameter(torch.empty(num_embeddings, embedding_dim, device=device, dtype=dtype))
        torch.nn.init.trunc_normal_(tensor=self.embeddings, mean=0, std=1, a=-3, b=3)

    # token_ids: torch.LongTensor (batch_size, sequence_length)
    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        return self.embeddings[token_ids]

In [6]:
embedding = Embedding(256, 1024)
embedding(torch.randint(0, 256, (2, 5)))

tensor([[[-1.8218,  0.4365, -0.1584,  ..., -0.0875, -0.2010,  0.4151],
         [ 0.1251, -1.5636, -0.4643,  ..., -0.5208, -1.0659,  1.0125],
         [ 1.0135,  0.9980,  0.8817,  ..., -1.4450, -0.4669, -0.9772],
         [ 0.7899,  0.0876,  1.4259,  ..., -1.5646,  1.7745,  1.5520],
         [-0.3790, -1.2074,  0.0455,  ...,  0.4071, -0.8746, -0.1624]],

        [[-0.8840, -0.0239,  1.4791,  ..., -1.6396,  1.4908,  1.1506],
         [ 0.8030, -0.1216, -0.0686,  ...,  2.0665, -0.4722,  0.3724],
         [-1.3255, -1.0503,  0.3606,  ...,  0.0655,  0.9265, -1.1048],
         [ 0.1127,  0.9288, -0.7761,  ..., -0.9604, -0.3848,  0.2688],
         [ 0.2807, -0.0189, -1.1746,  ..., -0.8357,  1.4001, -0.9996]]],
       grad_fn=<IndexBackward0>)

In [7]:
import torch

class RMSNorm(torch.nn.Module):

    gain: torch.Tensor # (d_model, )
    eps: float
    
    def __init__(self, d_model: int, eps: float = 1e-5, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.gain = torch.nn.parameter.Parameter(torch.ones(d_model, device=device, dtype=dtype))
        self.eps = eps

    # x (batch_size, sequence_length, d_model)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        in_dtype = x.dtype
        x = x.to(torch.float32)
        x_sqrd_mean = x.pow(2).mean(dim=-1, keepdim=True) # (batch_size, sequence_length, 1)
        rms = torch.sqrt(x_sqrd_mean + self.eps) # (batch_size, sequence_length, 1)
        # (batch_size, sequence_length, d_model) * (d_model, ) / (batch_size, sequence_length, 1)
        result = x * self.gain / rms # (batch_size, sequence_length, d_model)
        return result.to(in_dtype)


In [8]:
rms_norm = RMSNorm(1024)
rms_norm(torch.randn(1024))

tensor([-0.9612,  1.6646,  0.2377,  ..., -0.6987, -1.1426,  0.2234],
       grad_fn=<DivBackward0>)

In [9]:
import torch

class SwiGLU(torch.nn.Module):
    w_1: Linear # (d_ff, d_model)
    w_2: Linear # (d_model, d_ff)
    w_3: Linear # (d_ff, d_model)

    def __init__(self, d_model: int, d_ff: int | None = None, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        if d_ff is None:
            d_ff = round(8 * d_model / 3 / 64) * 64
        self.w_1 = Linear(d_model, d_ff, device, dtype)
        self.w_2 = Linear(d_ff, d_model, device, dtype)
        self.w_3 = Linear(d_model, d_ff, device, dtype)

    # x (batch_size, sequence_length, d_model)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w_2(self._silu(self.w_1(x)) * self.w_3(x))

    def _silu(self, x: torch.Tensor) -> torch.Tensor:
        return x * torch.sigmoid(x)

In [10]:
d_model: int = 1024
swiglu = SwiGLU(d_model)
swiglu(torch.randn(256, d_model))

tensor([[ 0.1718, -0.3284,  0.1140,  ...,  0.5165,  0.6141, -0.0409],
        [-0.0876,  0.2554,  0.4938,  ..., -0.3187,  0.1295, -0.0073],
        [ 0.3570, -0.4998, -0.0252,  ..., -0.5709, -0.1115,  0.3816],
        ...,
        [-0.5164,  0.1995, -0.3835,  ..., -0.3762,  0.1083,  0.0407],
        [ 0.1197,  0.6331,  0.0909,  ...,  0.4049,  0.2519, -0.0423],
        [-0.1866,  0.1819,  0.1874,  ...,  0.7727,  0.3563,  0.0624]],
       grad_fn=<ViewBackward0>)

In [11]:
import torch
from einops import rearrange

class RotaryPositionalEmbedding(torch.nn.Module):

    def __init__(self, theta: float, d_k: int, max_seq_len: int, device: torch.device | None = None):
        super().__init__()
        k = torch.arange(0, d_k, 2, device=device)
        positions = torch.arange(0, max_seq_len, 1, device=device).reshape(max_seq_len, 1)
        angles = positions / theta ** (k / d_k)
        self.register_buffer('sin', torch.sin(angles), persistent=False)
        self.register_buffer('cos', torch.cos(angles), persistent=False)

    # x (batch, seq_len, d_k), token_positions (batch, seq_len)
    def forward(self, x: torch.Tensor, token_positions: torch.Tensor) -> torch.Tensor:
        x = rearrange(x, "... seq_len (half two) -> ... seq_len half two", two=2)
        a = x[..., 0] # (..., seq, half)
        b = x[..., 1] # (..., seq, half)
        cos = self.cos[token_positions]
        sin = self.sin[token_positions]
        a_out = a * cos - b * sin
        b_out = a * sin + b * cos
        x_out = torch.stack([a_out, b_out], dim=-1) # (..., seq_len, d_k/2 2)
        x_out = rearrange(x_out, "... seq_len half two -> ... seq_len (half two)")
        return x_out

In [12]:
d_k = 4
theta = 10000
max_seq_len = 2
x = torch.tensor([[1., 0., 1., 0.], [1., 0., 1., 0.]])
rope = RotaryPositionalEmbedding(theta, d_k, max_seq_len) # theta, d_k, max_seq_len
rope.cos.shape # seq_len, d_k/2
token_positions = torch.arange(max_seq_len)
rope(x, token_positions)

tensor([[1.0000, 0.0000, 1.0000, 0.0000],
        [0.5403, 0.8415, 0.9999, 0.0100]])

In [13]:
import torch

def softmax(x: torch.Tensor, dim: int) -> torch.Tensor:
    x_max = x.max(dim=-1, keepdim=True).values
    x_norm = x.subtract(x_max)
    return x_norm.exp() / x_norm.exp().sum(dim=dim, keepdim=True)

In [14]:
x = torch.randn(2, 3)
print(x)
y = x.max(dim=-1, keepdim=True).values
print(y)
print(x.subtract(y))
print(softmax(x, -1))

tensor([[ 0.5181,  1.4804,  1.2273],
        [-2.3413, -0.4667, -0.0123]])
tensor([[ 1.4804],
        [-0.0123]])
tensor([[-0.9623,  0.0000, -0.2531],
        [-2.3289, -0.4543,  0.0000]])
tensor([[0.1770, 0.4633, 0.3597],
        [0.0562, 0.3665, 0.5773]])


In [15]:
torch.allclose(torch.softmax(x, dim=-1), softmax(x, -1))

True

In [16]:
import torch 
from jaxtyping import Bool, Float

d_k = 4
d_v = 3
keys = 2
queries = 2

Q: Float[torch.Tensor, " ... queries d_k"] = torch.randn(keys, d_k)
K: Float[torch.Tensor, " ... keys d_k"] = torch.randn(queries, d_k)
V: Float[torch.Tensor, " ... keys d_v"] = torch.randn(keys, d_v)
mask: Bool[torch.Tensor, " ... queries keys"] | None = (torch.randn(queries, keys).uniform_() > 0.8)

In [17]:
import torch
from jaxtyping import Bool, Float
from einops import einsum
import math

def scaled_dot_product_attention(
        Q: Float[torch.Tensor, " ... queries d_k"],
        K: Float[torch.Tensor, " ... keys d_k"],
        V: Float[torch.Tensor, " ... keys d_v"],
        mask: Bool[torch.Tensor, " ... queries keys"] | None = None
) -> Float[torch.Tensor, " ... seq_len d_v"]:
    d_k = Q.shape[-1]
    qk = einsum(Q, K, "... queries d_k, ... keys d_k -> ... queries keys") / math.sqrt(d_k)
    if mask is not None:
        qk = qk.masked_fill(~mask, float("-Inf"))
    qk = softmax(qk, dim=-1)
    return einsum(qk, V, "... queries keys, ... keys d_v -> ... queries d_v")

In [18]:
scaled_dot_product_attention(Q, K, V, mask)

tensor([[nan, nan, nan],
        [nan, nan, nan]])

In [19]:
torch.allclose(torch.nn.functional.scaled_dot_product_attention(Q, K, V), scaled_dot_product_attention(Q, K, V))

True

In [20]:
import torch
from einops import einsum, rearrange

class CasualMultiHeadSelfAttention(torch.nn.Module):

    device: torch.device | None
    d_k: int
    d_v: int
    d_model: int
    num_heads: int
    weights_q: Linear # (num_heads * d_k, d_model) = (hd_k, d_model)
    weights_k: Linear # (num_heads * d_k, d_model) = (hd_k, d_model)
    weights_v: Linear # (num_heads * d_v, d_model) = (hd_v, d_model)
    weights_o: Linear # (d_model, num_heads * d_v) = (d_model, hd_v)
    rope: RotaryPositionalEmbedding | None

    def __init__(self, d_model: int, num_heads: int, max_seq_len: int | None = None, theta: float | None = None, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        if d_model % num_heads != 0:
            raise Exception("d_model must be divisible by num_heads!")
        self.device = device
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = round(d_model / num_heads)
        self.d_v = self.d_k
        self.weights_q = Linear(d_model, num_heads * self.d_k, device, dtype)
        self.weights_k = Linear(d_model, num_heads * self.d_k, device, dtype)
        self.weights_v = Linear(d_model, num_heads * self.d_v, device, dtype)
        self.weights_o = Linear(num_heads * self.d_v, d_model, device, dtype)
        if max_seq_len is not None and theta is not None:
            self.rope = RotaryPositionalEmbedding(theta, self.d_k, max_seq_len)
        else:
            self.rope = None

    # x (..., seq_len, d_model)
    def forward(self, x: torch.Tensor, token_positions: torch.Tensor | None = None) -> torch.Tensor:
        seq_len = x.shape[-2]
        casual_mask = (torch.tril(torch.ones(seq_len, seq_len, device=self.device)) == 1)
        if token_positions is None:
            token_positions = torch.arange(seq_len, device=self.device)
        wq_x = self.weights_q(x)  # (..., seq_len, hd_k)
        wk_x = self.weights_k(x)  # (..., seq_len, hd_k)
        wv_x = self.weights_v(x)  # (..., seq_len, hd_v)
        wq_x_i = rearrange(wq_x, "... seq_len (h d_k) -> ... h seq_len d_k", h=self.num_heads)  # (..., h, seq_len, d_k)
        wk_x_i = rearrange(wk_x, "... seq_len (h d_k) -> ... h seq_len d_k", h=self.num_heads)  # (..., h, seq_len, d_k)
        wv_x_i = rearrange(wv_x, "... seq_len (h d_v) -> ... h seq_len d_v", h=self.num_heads)  # (..., h, seq_len, d_v)
        if self.rope:
            wq_x_i = self.rope(wq_x_i, token_positions)
            wk_x_i = self.rope(wk_x_i, token_positions)
        result = scaled_dot_product_attention(wq_x_i, wk_x_i, wv_x_i, casual_mask)  # (..., h, seq_len, d_v)
        result = rearrange(result, "... h seq_len d_v -> ... seq_len (h d_v)")
        return self.weights_o(result)


In [21]:
import torch
from einops import einsum, rearrange

class CasualMultiHeadSelfAttentionOptimized(torch.nn.Module):

    device: torch.device | None
    d_k: int
    d_v: int
    d_model: int
    num_heads: int
    weights_qkv: Linear # (hd_k + hd_k + hd_v, d_model)
    weights_o: Linear # (d_model, num_heads * d_v) = (d_model, hd_v)
    rope: RotaryPositionalEmbedding | None

    def __init__(self, d_model: int, num_heads: int, max_seq_len: int | None = None, theta: float | None = None, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        if d_model % num_heads != 0:
            raise Exception("d_model must be divisible by num_heads!")
        self.device = device
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = round(d_model / num_heads)
        self.d_v = self.d_k
        self.weights_qkv = Linear(d_model, num_heads * self.d_k + num_heads * self.d_k + num_heads * self.d_v, device, dtype)
        self.weights_o = Linear(num_heads * self.d_v, d_model, device, dtype)
        if max_seq_len is not None and theta is not None:
            self.rope = RotaryPositionalEmbedding(theta, self.d_k, max_seq_len)
        else:
            self.rope = None

    # x (..., seq_len, d_model)
    def forward(self, x: torch.Tensor, token_positions: torch.Tensor | None = None) -> torch.Tensor:
        seq_len = x.shape[-2]
        casual_mask = (torch.tril(torch.ones(seq_len, seq_len, device=self.device)) == 1)
        if token_positions is None:
            token_positions = torch.arange(seq_len, device=self.device)
        w_x = self.weights_qkv(x)  # (..., seq_len, hd_k + hd_k + hd_v)
        wq_x = w_x[..., :self.num_heads * self.d_k]
        wk_x = w_x[..., self.num_heads * self.d_k:2 * self.num_heads * self.d_k]
        wv_x = w_x[..., 2 * self.num_heads * self.d_k:]
        wq_x_i = rearrange(wq_x, "... seq_len (h d_k) -> ... h seq_len d_k", h=self.num_heads)  # (..., h, seq_len, d_k)
        wk_x_i = rearrange(wk_x, "... seq_len (h d_k) -> ... h seq_len d_k", h=self.num_heads)  # (..., h, seq_len, d_k)
        wv_x_i = rearrange(wv_x, "... seq_len (h d_v) -> ... h seq_len d_v", h=self.num_heads)  # (..., h, seq_len, d_v)
        if self.rope:
            wq_x_i = self.rope(wq_x_i, token_positions)
            wk_x_i = self.rope(wk_x_i, token_positions)
        result = scaled_dot_product_attention(wq_x_i, wk_x_i, wv_x_i, casual_mask)  # (..., h, seq_len, d_v)
        result = rearrange(result, "... h seq_len d_v -> ... seq_len (h d_v)")
        return self.weights_o(result)


In [22]:
d_model = 16
num_heads = 4
max_seq_len = 8
theta = 10000
attention = CasualMultiHeadSelfAttention(d_model, num_heads)
print(attention)
attention_with_rope = CasualMultiHeadSelfAttention(d_model, num_heads, max_seq_len, theta)
attention_with_rope_optimized = CasualMultiHeadSelfAttentionOptimized(d_model, num_heads, max_seq_len, theta)
print(attention_with_rope)
x = torch.randn(max_seq_len, d_model)
print(attention(x))
print(attention_with_rope(x))
print(attention_with_rope_optimized(x))

CasualMultiHeadSelfAttention(
  (weights_q): Linear()
  (weights_k): Linear()
  (weights_v): Linear()
  (weights_o): Linear()
)
CasualMultiHeadSelfAttention(
  (weights_q): Linear()
  (weights_k): Linear()
  (weights_v): Linear()
  (weights_o): Linear()
  (rope): RotaryPositionalEmbedding()
)
tensor([[ 0.4709,  0.2845, -0.9287, -0.1104, -1.1443,  0.4386, -1.7671,  0.4993,
         -0.4976, -0.2421,  1.5088,  0.4012, -0.8415, -0.5850,  0.8671,  1.3535],
        [ 0.9752,  0.8033,  0.1015, -0.9919, -0.5291, -0.1349, -2.6826,  1.2701,
         -0.4749, -0.9312,  1.8656, -0.5203, -1.5387, -0.9182,  1.4409,  0.1010],
        [ 0.6617,  0.9873, -0.4079, -1.6581,  0.0118, -0.2742, -2.3651,  0.5548,
         -0.3504, -1.1079,  1.6279, -0.5992, -1.5853,  0.1172,  2.6262,  0.0353],
        [ 0.3794,  1.0088,  0.1360, -0.9345, -0.2591,  0.1024, -0.9600,  0.8473,
         -0.2801, -0.8670,  1.0684,  0.4068, -1.6191, -0.7222,  1.5006,  0.2427],
        [ 0.8498,  1.0074,  0.0534, -1.2042, -0.4584, 

In [23]:
import torch

class TransformerBlock(torch.nn.Module):

    norm_attention: RMSNorm
    attention: CasualMultiHeadSelfAttention

    feed_forward: SwiGLU
    norm_feed_forward: RMSNorm

    def __init__(self, d_model: int, num_heads: int, d_ff: int | None = None, eps: float = 1e-5, max_seq_len: int | None = None, theta: float | None = None, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.norm_attention = RMSNorm(d_model, eps, device=device, dtype=dtype)
        self.attention = CasualMultiHeadSelfAttention(d_model, num_heads, max_seq_len, theta, device=device, dtype=dtype)
        self.norm_feed_forward = RMSNorm(d_model, eps, device=device, dtype=dtype)
        self.feed_forward = SwiGLU(d_model, d_ff, device=device, dtype=dtype)

    def forward(self, x: torch.Tensor, token_positions: torch.Tensor | None = None) -> torch.Tensor:
        y = x + self.attention(self.norm_attention(x), token_positions)
        y = y + self.feed_forward(self.norm_feed_forward(y))
        return y

In [24]:
d_model = 16
num_heads = 4
d_ff = round(8 * d_model / 3 / 64) * 64
transformer = TransformerBlock(d_model, num_heads, d_ff)
print(transformer)
x = torch.randn(4, d_model)
transformer(x)

TransformerBlock(
  (norm_attention): RMSNorm()
  (attention): CasualMultiHeadSelfAttention(
    (weights_q): Linear()
    (weights_k): Linear()
    (weights_v): Linear()
    (weights_o): Linear()
  )
  (norm_feed_forward): RMSNorm()
  (feed_forward): SwiGLU(
    (w_1): Linear()
    (w_2): Linear()
    (w_3): Linear()
  )
)


tensor([[ 2.6629, -1.8332, -2.9420, -0.8070,  0.1429,  0.4046, -1.5794,  4.4135,
          0.1757,  0.7458, -0.3910,  2.1432, -2.2251,  2.2075,  1.3770,  1.5599],
        [-0.8240, -0.9258,  0.9380, -1.3365, -1.0346, -0.4988,  0.8230,  3.3265,
          0.1156,  3.1086,  0.0978,  2.1620, -0.1302,  0.3443,  0.7098,  2.0197],
        [ 1.1281,  0.6129,  0.1940, -0.7986,  1.2345,  1.2519,  1.0982, -0.3614,
         -0.7774, -1.7053, -0.3268, -1.4676, -3.5788,  1.0994,  0.0155,  0.4486],
        [-0.4757, -0.4873, -1.8452,  1.2825,  1.4958,  1.6650,  1.2823,  0.0850,
          3.3467, -0.1532, -1.6105, -0.0527,  1.8910,  0.0122, -1.1083,  0.1146]],
       grad_fn=<AddBackward0>)

In [25]:
import torch

class TransformerLM(torch.nn.Module):

    token_embeddings: Embedding
    layers: torch.nn.ModuleList
    ln_final: RMSNorm
    lm_head: Linear

    def __init__(self, vocab_size: int, context_length: int, num_layers: int, d_model: int, num_heads: int, d_ff: int | None = None, eps: float = 1e-5, theta: float | None = None, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.token_embeddings = Embedding(num_embeddings=vocab_size, embedding_dim=d_model, device=device, dtype=dtype)
        self.layers = torch.nn.ModuleList(TransformerBlock(d_model, num_heads, d_ff, eps, context_length, theta, device, dtype) for _ in range(num_layers))
        self.ln_final = RMSNorm(d_model, eps, device=device, dtype=dtype)
        self.lm_head = Linear(d_model, vocab_size, device=device, dtype=dtype)
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.token_embeddings(x)
        for transformer in self.layers:
            y = transformer(y)
        y = self.ln_final(y)
        y = self.lm_head(y)
        y = softmax(y, -1)
        return y

In [26]:
vocab_size = 256
context_length = 1000
num_layers = 2
d_model = 16
num_heads = 4
d_ff = round(8 * d_model / 3 / 64) * 64
eps: float = 1e-5
theta = 10000

transformer_lm = TransformerLM(vocab_size, context_length, num_layers, d_model, num_heads, d_ff, eps, theta)
print(transformer_lm)

x = torch.randint(0, 255, (4, 6))
y = transformer_lm(x)
print(y)

TransformerLM(
  (token_embeddings): Embedding()
  (layers): ModuleList(
    (0-1): 2 x TransformerBlock(
      (norm_attention): RMSNorm()
      (attention): CasualMultiHeadSelfAttention(
        (weights_q): Linear()
        (weights_k): Linear()
        (weights_v): Linear()
        (weights_o): Linear()
        (rope): RotaryPositionalEmbedding()
      )
      (norm_feed_forward): RMSNorm()
      (feed_forward): SwiGLU(
        (w_1): Linear()
        (w_2): Linear()
        (w_3): Linear()
      )
    )
  )
  (ln_final): RMSNorm()
  (lm_head): Linear()
)
tensor([[[0.0030, 0.0037, 0.0047,  ..., 0.0076, 0.0053, 0.0033],
         [0.0030, 0.0046, 0.0046,  ..., 0.0098, 0.0042, 0.0048],
         [0.0038, 0.0060, 0.0052,  ..., 0.0087, 0.0038, 0.0041],
         [0.0031, 0.0039, 0.0052,  ..., 0.0047, 0.0055, 0.0069],
         [0.0023, 0.0053, 0.0048,  ..., 0.0053, 0.0050, 0.0055],
         [0.0041, 0.0043, 0.0037,  ..., 0.0072, 0.0043, 0.0036]],

        [[0.0038, 0.0049, 0.0018,  ..., 0.

In [29]:
from torch import nn

def print_model_summary(model, input_size):
    def register_hook(module):
        def hook(module, input, output):
            class_name = str(module.__class__).split(".")[-1].split("'")[0]
            module_idx = len(summary)
            m_key = f"{class_name}-{module_idx+1}"
            summary[m_key] = {
                "input_shape": list(input[0].size()),
                "output_shape": list(output.size()),
                "nb_params": sum(p.numel() for p in module.parameters(recurse=False))
            }
        if not isinstance(module, nn.Sequential) and not isinstance(module, nn.ModuleList) and module != model:
            hooks.append(module.register_forward_hook(hook))

    summary = {}
    hooks = []
    model.apply(register_hook)
    with torch.no_grad():
        model(torch.zeros(1, *input_size, dtype=torch.long))

    for h in hooks:
        h.remove()

    print("----------------------------------------------------------------")
    line_new = "{:>20}  {:>25} {:>15}".format("Layer (type)", "Output Shape", "Param #")
    print(line_new)
    print("================================================================")
    total_params = 0
    for layer in summary:
        line_new = "{:>20}  {:>25} {:>15}".format(
            layer,
            str(summary[layer]["output_shape"]),
            "{0:,}".format(summary[layer]["nb_params"])
        )
        total_params += summary[layer]["nb_params"]
        print(line_new)
    print("================================================================")
    print(f"Total params: {total_params:,}")
    print("----------------------------------------------------------------")

# Example usage
print_model_summary(transformer_lm, (context_length, ))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
         Embedding-1              [1, 1000, 16]           4,096
           RMSNorm-2              [1, 1000, 16]              16
            Linear-3              [1, 1000, 16]             256
            Linear-4              [1, 1000, 16]             256
            Linear-5              [1, 1000, 16]             256
RotaryPositionalEmbedding-6            [1, 4, 1000, 4]               0
RotaryPositionalEmbedding-7            [1, 4, 1000, 4]               0
            Linear-8              [1, 1000, 16]             256
CasualMultiHeadSelfAttention-9              [1, 1000, 16]               0
          RMSNorm-10              [1, 1000, 16]              16
           Linear-11              [1, 1000, 64]           1,024
           Linear-12              [1, 1000, 64]           1,024
           Linear-13              [1, 1000, 16]           1,024
           SwiG

In [30]:
total = sum(p.numel() for p in transformer_lm.parameters())
total

16464

In [31]:
vocab_size = 50257
context_length = 1024
num_layers = 48
d_model = 1600
num_heads = 25
d_ff = 4288
eps: float = 1e-5
theta = 10000

transformer_assignment = TransformerLM(vocab_size, context_length, num_layers, d_model, num_heads, d_ff, eps, theta)
print(transformer_assignment)

print_model_summary(transformer_assignment, (context_length, ))

TransformerLM(
  (token_embeddings): Embedding()
  (layers): ModuleList(
    (0-47): 48 x TransformerBlock(
      (norm_attention): RMSNorm()
      (attention): CasualMultiHeadSelfAttention(
        (weights_q): Linear()
        (weights_k): Linear()
        (weights_v): Linear()
        (weights_o): Linear()
        (rope): RotaryPositionalEmbedding()
      )
      (norm_feed_forward): RMSNorm()
      (feed_forward): SwiGLU(
        (w_1): Linear()
        (w_2): Linear()
        (w_3): Linear()
      )
    )
  )
  (ln_final): RMSNorm()
  (lm_head): Linear()
)
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
         Embedding-1            [1, 1024, 1600]      80,411,200
           RMSNorm-2            [1, 1024, 1600]           1,600
            Linear-3            [1, 1024, 1600]       2,560,000
            Linear-4            [1, 1024, 1600]       2,560,000
            Linear-5            [1, 1024, 1600]

In [32]:
total = sum(p.numel() for p in transformer_assignment.parameters())
total

1640452800

In [6]:
import resource
resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024/1024

74.140625

In [3]:
import psutil, os
psutil.Process(os.getpid()).memory_info().rss / 1e9   # GB, current

0.027623424

In [76]:
import torch
from einops import rearrange

# inputs (batch_size, vocab_size)
# targets (batch_size)
def cross_entropy(inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    inputs_max = inputs.max(dim=-1, keepdim=True).values
    targets_per_batch = rearrange(targets, "batch -> batch 1")
    inputs_norm = inputs.subtract(inputs_max)
    target_logit = torch.gather(inputs_norm, -1, targets_per_batch)
    result = -1 * (target_logit - inputs_norm.exp().sum(dim=-1, keepdim=True).log())
    return result.mean()

In [77]:
inputs = torch.randn(16, 256)
print(f"inputs: {inputs.shape}")
targets = torch.randint(0, 255, (16, ))
print(f"targets: {targets.shape}")
cross_entropy(inputs, targets)

inputs: torch.Size([16, 256])
targets: torch.Size([16])


tensor(5.6894)

In [78]:
from collections.abc import Callable, Iterable
from typing import Optional
import torch
import math

class SGD(torch.optim.Optimizer):

    def __init__(self, params, lr=1e-3) -> None:
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr}
        super().__init__(params, defaults)

    def step(self, closure: Optional[Callable] = None) -> Optional[Callable]:
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group["lr"] # Get the learning rate.
            for p in group["params"]:
                if p.grad is None:
                    continue
                state = self.state[p] # Get state associated with p.
                t = state.get("t", 0) # Get iteration number from the state, or 0.
                grad = p.grad.data # Get the gradient of loss with respect to p.
                p.data -= lr / math.sqrt(t + 1) * grad # Update weight tensor in-place.
                state["t"] = t + 1 # Increment iteration number.
        return loss

In [82]:
for lr in [1e1, 1e2, 1e3]:
    print(f"lr: {lr}")
    weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
    opt = SGD([weights], lr=lr)
    for t in range(10):
        opt.zero_grad() # Reset the gradients for all learnable parameters.
        loss = (weights**2).mean() # Compute a scalar loss value.
        print(loss.cpu().item())
        loss.backward() # Run backward pass, which computes gradients.
        opt.step() # Run optimizer step.
    print("==========")

lr: 10.0
20.33173179626465
13.012308120727539
9.592118263244629
7.504806995391846
6.078894138336182
5.040099620819092
4.250655651092529
3.6323060989379883
3.136782169342041
2.7324860095977783
lr: 100.0
23.124834060668945
23.124828338623047
3.9675936698913574
0.09495346248149872
2.0848990724703516e-16
2.323749484393927e-18
7.824882379272437e-20
4.661336023456421e-21
3.9987952509053815e-22
4.443106009641736e-23
lr: 1000.0
21.267976760864258
7677.7392578125
1326066.375
147510720.0
11948368896.0
754078908416.0
38711945330688.0
1665553420779520.0
6.138874495080858e+16
1.9712609336866898e+18


In [83]:
weights.shape

torch.Size([10, 10])

In [119]:
import math
from typing import Optional, Callable, Iterable

import torch


class AdamW(torch.optim.Optimizer):

    def __init__(self, params: Iterable[torch.nn.Parameter], lr=1e-3, weight_decay=0.01, betas=(0.9, 0.999), eps=1e-8) -> None:
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr, "weight_decay": weight_decay, "betas": betas, "eps": eps}
        super().__init__(params, defaults)

    def step(self, closure: Optional[Callable] = None) -> Optional[Callable]:
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group["lr"]  # Get the learning rate.
            weight_decay = group["weight_decay"]
            betas = group["betas"]
            eps = group["eps"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                state = self.state[p]  # Get state associated with p.
                t = state.get("t", 1)  # Get iteration number from the state, or 0.
                m = state["m"] if "m" in state else torch.zeros_like(p)
                v = state["v"] if "v" in state else torch.zeros_like(p)
                grad = p.grad.data  # Get the gradient of loss with respect to p.
                lr_t = lr * (math.sqrt(1 - betas[1]**t)/(1-betas[0]**t)) # Compute adjusted 𝛼 for iteration 𝑡
                p.data -= lr * weight_decay * p.data # Apply weight decay
                m = betas[0] * m + (1 - betas[0]) * grad # Update the first moment estimate
                v = betas[1] * v + (1 - betas[1]) * grad**2 # Update the second moment estimate
                p.data -= lr_t * m / (v.sqrt() + eps)  # Apply moment-adjusted weight updates
                state["t"] = t + 1  # Increment iteration number.
                state["m"] = m
                state["v"] = v
        return loss


In [120]:

weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
print(weights.shape)
opt = AdamW([weights])
for t in range(10):
    opt.zero_grad() # Reset the gradients for all learnable parameters.
    loss = (weights**2).mean() # Compute a scalar loss value.
    print(loss.cpu().item())
    loss.backward() # Run backward pass, which computes gradients.
    opt.step() # Run optimizer step.
print("==========")

torch.Size([10, 10])
26.873544692993164
26.864717483520508
26.85589027404785
26.847068786621094
26.83824920654297
26.829429626464844
26.82061004638672
26.811796188354492
26.802988052368164
26.79418182373047


In [1]:
import torch
from typing import Optional
from cs336_basics.softmax import softmax


def sample_next_token(logits: torch.Tensor, temperature: float, top_p: Optional[float] = None, top_k: Optional[int] = None) -> int:
    if temperature == 0:
        return torch.argmax(logits)
    probabilities = softmax(logits, dim=0)
    values, indices = torch.sort(probabilities)
    if top_k:
        values[top_k:0] = 0
    if top_p:
        cum_sum = torch.cumsum(values, 0)
        difference = cum_sum - values
        mask = difference >= top_p
        values[mask] = 0
    next_token_id = torch.multinomial(values, num_samples=1)
    return indices[next_token_id].item()

In [2]:
logits = torch.randn(1, 1024,  10000)
print(logits.shape)
logits = logits[0][-1]
print(logits.shape)
next_token_id = sample_next_token(logits, 1.0, top_p=None, top_k=None)
print(f"Next token id: {next_token_id}")

torch.Size([1, 1024, 10000])
torch.Size([10000])
Next token id: 8378


In [3]:
probabilities = softmax(logits, dim=0)
values, indices = torch.sort(probabilities, descending=True)
values[3:] = 0
values

tensor([0.0030, 0.0020, 0.0016,  ..., 0.0000, 0.0000, 0.0000])

In [4]:
top_p = 0.9
probabilities = softmax(logits, dim=0)
values, indices = torch.sort(probabilities, descending=True)
print(values)
cum_sum = torch.cumsum(values, 0)
difference = cum_sum - values
mask1 = difference >= top_p
mask = cum_sum > top_p
print(mask)
print(torch.allclose(mask1, mask))
print((mask != mask1).sum())


tensor([2.9526e-03, 2.0458e-03, 1.5727e-03,  ..., 2.0683e-06, 1.9875e-06,
        1.4855e-06])
tensor([False, False, False,  ...,  True,  True,  True])
False
tensor(1)


In [5]:
top_p = 0.9
probabilities = softmax(logits, dim=0)
values, indices = torch.sort(probabilities, descending=True)
cum_sum = torch.cumsum(values, 0)
difference = cum_sum - values
mask = difference >= top_p
values[mask] = 0
values

tensor([0.0030, 0.0020, 0.0016,  ..., 0.0000, 0.0000, 0.0000])

In [2]:
indices: list[int] = [i * 256 for i in range(10)]
indices

[0, 256, 512, 768, 1024, 1280, 1536, 1792, 2048, 2304]

In [3]:
import numpy as np

In [5]:
arr = np.array(indices)
arr

array([   0,  256,  512,  768, 1024, 1280, 1536, 1792, 2048, 2304])

In [10]:
indices = np.array([i * 256 for i in range(10 * 64)])
batches = indices.reshape((10, 64))
batches

array([[     0,    256,    512,    768,   1024,   1280,   1536,   1792,
          2048,   2304,   2560,   2816,   3072,   3328,   3584,   3840,
          4096,   4352,   4608,   4864,   5120,   5376,   5632,   5888,
          6144,   6400,   6656,   6912,   7168,   7424,   7680,   7936,
          8192,   8448,   8704,   8960,   9216,   9472,   9728,   9984,
         10240,  10496,  10752,  11008,  11264,  11520,  11776,  12032,
         12288,  12544,  12800,  13056,  13312,  13568,  13824,  14080,
         14336,  14592,  14848,  15104,  15360,  15616,  15872,  16128],
       [ 16384,  16640,  16896,  17152,  17408,  17664,  17920,  18176,
         18432,  18688,  18944,  19200,  19456,  19712,  19968,  20224,
         20480,  20736,  20992,  21248,  21504,  21760,  22016,  22272,
         22528,  22784,  23040,  23296,  23552,  23808,  24064,  24320,
         24576,  24832,  25088,  25344,  25600,  25856,  26112,  26368,
         26624,  26880,  27136,  27392,  27648,  27904,  28160,

In [12]:
for idx in range(0, len(indices), 64):
    batch = indices[idx : (idx + 64)]
    print(batch)

[    0   256   512   768  1024  1280  1536  1792  2048  2304  2560  2816
  3072  3328  3584  3840  4096  4352  4608  4864  5120  5376  5632  5888
  6144  6400  6656  6912  7168  7424  7680  7936  8192  8448  8704  8960
  9216  9472  9728  9984 10240 10496 10752 11008 11264 11520 11776 12032
 12288 12544 12800 13056 13312 13568 13824 14080 14336 14592 14848 15104
 15360 15616 15872 16128]
[16384 16640 16896 17152 17408 17664 17920 18176 18432 18688 18944 19200
 19456 19712 19968 20224 20480 20736 20992 21248 21504 21760 22016 22272
 22528 22784 23040 23296 23552 23808 24064 24320 24576 24832 25088 25344
 25600 25856 26112 26368 26624 26880 27136 27392 27648 27904 28160 28416
 28672 28928 29184 29440 29696 29952 30208 30464 30720 30976 31232 31488
 31744 32000 32256 32512]
[32768 33024 33280 33536 33792 34048 34304 34560 34816 35072 35328 35584
 35840 36096 36352 36608 36864 37120 37376 37632 37888 38144 38400 38656
 38912 39168 39424 39680 39936 40192 40448 40704 40960 41216 41472 41728

In [14]:
import torch
G = torch.tensor([[1.51, 1.49], [1.49, 1.51]])
print(torch.linalg.svdvals(G))

tensor([3.0000, 0.0200])


In [16]:
U, S, Vh = torch.linalg.svd(G)
print(U)
print(S)
print(Vh)
O = U @ Vh
print(O)
print(torch.linalg.svdvals(O))

tensor([[-0.7071, -0.7071],
        [-0.7071,  0.7071]])
tensor([3.0000, 0.0200])
tensor([[-0.7071, -0.7071],
        [-0.7071,  0.7071]])
tensor([[ 1.0000e+00, -2.9802e-08],
        [-2.9802e-08,  1.0000e+00]])
tensor([1.0000, 1.0000])


In [25]:
import torch

a = 3.4445
b = -4.7750
c = 2.0315

G = torch.randn(64, 1) @ torch.randn(1, 32) + 0.01 * torch.randn(64, 32)
eps = 1e-7
X = G / (G.pow(2).sum().sqrt() + eps)
for _ in range(5):
    A = X @ X.T
    X = a * X + (b * A + c * A @ A) @ X
print(torch.linalg.svdvals(G))
print(torch.linalg.svdvals(X))


tensor([5.3749e+01, 1.2906e-01, 1.2174e-01, 1.1870e-01, 1.1169e-01, 1.0748e-01,
        1.0652e-01, 1.0290e-01, 9.6408e-02, 9.3026e-02, 9.0873e-02, 8.5521e-02,
        8.3312e-02, 8.1618e-02, 8.0728e-02, 7.7442e-02, 7.3514e-02, 7.2649e-02,
        6.5920e-02, 6.5372e-02, 6.1270e-02, 5.8500e-02, 5.7479e-02, 5.2596e-02,
        5.1546e-02, 4.9616e-02, 4.5414e-02, 4.5043e-02, 4.1919e-02, 3.6899e-02,
        3.5497e-02, 2.4124e-02])
tensor([0.9792, 0.9414, 0.9249, 0.8850, 0.8599, 0.8540, 0.8316, 0.7896, 0.7670,
        0.7524, 0.7150, 0.6993, 0.6965, 0.6870, 0.6806, 0.6564, 0.6270, 0.6204,
        0.5684, 0.5641, 0.5316, 0.5093, 0.5010, 0.4610, 0.4523, 0.4363, 0.4010,
        0.3979, 0.3713, 0.3282, 0.3160, 0.2163])
